In [ ]:
from pathlib import Path
import os

BASE_DIR = Path("/path/to/nnunet")
DATASET_ROOT = BASE_DIR / "nnUNet_raw" / "Dataset002_SPIDER"
IMAGES_TR = DATASET_ROOT / "imagesTr"
LABELS_TR = DATASET_ROOT / "labelsTr"

os.environ["nnUNet_raw"] = str(BASE_DIR / "nnUNet_raw")
os.environ["nnUNet_preprocessed"] = str(BASE_DIR / "nnUNet_preprocessed")
os.environ["nnUNet_results"] = str(BASE_DIR / "nnUNet_results")
os.environ["TMPDIR"] = str(BASE_DIR / "tmp")
os.environ["TMP"] = str(BASE_DIR / "tmp")

for directory in [IMAGES_TR, LABELS_TR, BASE_DIR / "nnUNet_preprocessed", BASE_DIR / "nnUNet_results"]:
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import shutil

image_src = "/path/to/your/data/Spiderdataset/images/images"
mask_src  = "/path/to/your/data/Spiderdataset/masks/masks" 

image_dst = str(IMAGES_TR) 
mask_dst  = str(LABELS_TR)

os.makedirs(image_dst, exist_ok=True)
os.makedirs(mask_dst, exist_ok=True)

t2_images = glob.glob(os.path.join(image_src, "*t2*.mha"))
t2_masks = glob.glob(os.path.join(mask_src, "*t2*.mha"))

for img in t2_images:

    name = os.path.basename(img)
    mask = os.path.join(mask_src, name)

    if os.path.exists(mask):

        shutil.copy(img, image_dst)
        shutil.copy(mask, mask_dst)

print("T2 images and masks copied.")

In [ ]:
import os, glob
import SimpleITK as sitk

# Source folders
image_src = str(IMAGES_TR)
mask_src  = str(LABELS_TR)

# Destination folders (same as source, overwrite with .nii.gz)
dst_images = image_src
dst_masks  = mask_src

# Convert T2 images
for img in glob.glob(os.path.join(image_src, "*_t2.mha")):
    case_id = os.path.basename(img).split("_")[0]   # "1" from "1_t2.mha"

    # Read and convert image
    image = sitk.ReadImage(img)
    out_img = os.path.join(dst_images, f"{case_id}_0000.nii.gz")
    sitk.WriteImage(image, out_img)

    # Read and convert mask (same filename as image)
    mask_file = os.path.join(mask_src, f"{case_id}_t2.mha")
    if os.path.exists(mask_file):
        mask = sitk.ReadImage(mask_file)
        out_mask = os.path.join(dst_masks, f"{case_id}.nii.gz")
        sitk.WriteImage(mask, out_mask)
        print(f"Converted {img} → {out_img}, {mask_file} → {out_mask}")
    else:
        print(f"No mask found for {img}")

print("Conversion complete.")

In [ ]:
import nibabel as nib
import numpy as np
import glob, os


# Output folder where corrected labels will be saved
label_dir = str(LABELS_TR)

# Mapping: vertebrae 1–5 stay as 1–5, discs 201–206 → 6–11
mapping = {
    1: 1,   # L1
    2: 2,   # L2
    3: 3,   # L3
    4: 4,   # L4
    5: 5,   # L5
    201: 6, # T12-L1 disc
    202: 7, # L1-L2 disc
    203: 8, # L2-L3 disc
    204: 9, # L3-L4 disc
    205: 10,# L4-L5 disc
    206: 11 # L5-S1 disc
}

for lbl_path in glob.glob(os.path.join(label_dir, "*.nii.gz")):
    img = nib.load(lbl_path)
    data = img.get_fdata().astype(np.int16)

    # Apply mapping
    new_data = np.zeros_like(data)
    for old_val, new_val in mapping.items():
        new_data[data == old_val] = new_val

    # Save back in place (overwrite)
    new_img = nib.Nifti1Image(new_data, img.affine, img.header)
    nib.save(new_img, lbl_path)

    print("Mapped:", os.path.basename(lbl_path))

In [ ]:
import nibabel as nib
import numpy as np
import glob
import os

label_dir = str(LABELS_TR)

valid_labels = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11}

for lf in glob.glob(os.path.join(label_dir, "*.nii*")):
    img = nib.load(lf)
    data = img.get_fdata().astype(np.int16)

    # Set all unwanted labels to background
    data[~np.isin(data, list(valid_labels))] = 0

    nib.save(
        nib.Nifti1Image(data, img.affine, img.header),
        lf
    )

print("Labels sanitized to L1–L5 and T12/L1 - L5/S1 only.")

In [ ]:
import os
import re
import glob
import json

# Update this path to your new dataset location
DATASET_ROOT = str(DATASET_ROOT)
IMAGES_TR = os.path.join(DATASET_ROOT, "imagesTr")
LABELS_TR = os.path.join(DATASET_ROOT, "labelsTr")
DATASET_JSON = os.path.join(DATASET_ROOT, "dataset.json")

pairs = []
for img_path in sorted(glob.glob(os.path.join(IMAGES_TR, "*_0000.nii.gz"))):
    case_id = os.path.basename(img_path).split("_")[0]
    lbl_path = os.path.join(LABELS_TR, f"{case_id}.nii.gz")
    if os.path.exists(lbl_path):
        rel_img = os.path.relpath(img_path, DATASET_ROOT)
        rel_lbl = os.path.relpath(lbl_path, DATASET_ROOT)
        pairs.append({"image": rel_img, "label": rel_lbl})
    else:
        print(f"WARNING: No label found for {case_id}")


# Build dataset.json (nnU-Net v2 format)
dataset = {
    "name": "Dataset002_SPIDER",
    "description": "SPIDER lumbar vertebrae + disc segmentation (T2 MRI)",
    "tensorImageSize": "3D",
    "reference": "SPIDER dataset (Zenodo); locally reformatted dataset",
    "licence": "Academic",
    "release": "2026-03-11",
    "channel_names": {"0": "MRI"},
    "labels": {
        "background": 0,
        "L1": 1,
        "L2": 2,
        "L3": 3,
        "L4": 4,
        "L5": 5,
        "T12-L1_disc": 6,
        "L1-L2_disc": 7,
        "L2-L3_disc": 8,
        "L3-L4_disc": 9,
        "L4-L5_disc": 10,
        "L5-S1_disc": 11
    },
    "numTraining": len(pairs),
    "numTest": 0,
    "training": pairs,
    "test": [],
    "file_ending": ".nii.gz"
}

# Save dataset.json
with open(DATASET_JSON, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"✅ Wrote dataset.json at: {DATASET_JSON}")

In [ ]:
!nnUNetv2_plan_and_preprocess -d 2 --verify_dataset_integrity

In [ ]:
!CUDA_VISIBLE_DEVICES=0 nnUNetv2_train Dataset002_SPIDER 3d_fullres 0